In [ ]:
# # Initial Imports and Variables
import numpy as np
import torch
import torchvision
import time
import matplotlib.pyplot as plt

import matplotlib
import sns
import os

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

load_dir= "./Data/"
results_directory="./Results/"
RANDOM_STATE=2025

class_list=['Floor-Bite', 'Floor-Explore', 'Floor-Poke','Stand-Bite', 'Stand-Eat', 'Stand-Explore', 'Stand-Poke']

In [ ]:
%run Utilities.py
from IPython.core.magic import register_cell_magic
from IPython import get_ipython

@register_cell_magic
def skip(line, cell):
    return

In [ ]:
import torch

if torch.cuda.is_available():
    idx = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(idx)

    print("GPU name:", props.name)
    print("Total memory (GB):", round(props.total_memory / 1024**3, 2))
    print("SMs:", props.multi_processor_count)
    print("Compute capability:", f"{props.major}.{props.minor}")
    print("Max threads per SM:", props.max_threads_per_multi_processor)
else:
    print("CUDA not available")


# Speed Evaluation Function

In [ ]:
import torch
import time
from ptflops import get_model_complexity_info


def evaluate_model_cost(
    model,
    input_size=(3, 128, 128),
    batch_size=32,
    clip_size=1,              # =1 for images, =num_frames for video
    n_warmup=20,
    n_iters=100,
    device="cuda"
):
    """
    Definitions:
      - 1 sample = 1 image OR 1 video clip
      - Samples/s = clips/s for video models
      - FPS = samples/s × clip_size
        (for images, clip_size=1 ⇒ FPS = samples/s)
    """

    assert clip_size >= 1, "clip_size must be >= 1"

    if clip_size == 1 and len(input_size) > 3:
        print("Warning: clip_size=1 but input_size suggests a video model.")

    model = model.to(device).eval()

    # ---- MACs / Params per sample ----
    macs, params = get_model_complexity_info(
        model,
        input_size,
        as_strings=False,
        print_per_layer_stat=False,
        verbose=False
    )

    gmacs_per_sample = macs / 1e9
    gmacs_total = gmacs_per_sample * batch_size
    params_m = params / 1e6

    # ---- Samples/s (batch = 1) ----
    x1 = torch.randn(1, *input_size).to(device)

    with torch.no_grad():
        for _ in range(n_warmup):
            model(x1)

        if device == "cuda":
            torch.cuda.synchronize()

        start = time.time()
        for _ in range(n_iters):
            model(x1)

        if device == "cuda":
            torch.cuda.synchronize()

        elapsed = time.time() - start
        samples_s_batch1 = n_iters / elapsed

    # ---- Samples/s (batch = N) ----
    xb = torch.randn(batch_size, *input_size).to(device)

    with torch.no_grad():
        for _ in range(n_warmup):
            model(xb)

        if device == "cuda":
            torch.cuda.synchronize()

        start = time.time()
        for _ in range(n_iters):
            model(xb)

        if device == "cuda":
            torch.cuda.synchronize()

        elapsed = time.time() - start
        samples_s_batchN = (n_iters * batch_size) / elapsed

    # ---- FPS (derived) ----
    fps_batch1 = samples_s_batch1 * clip_size
    fps_batchN = samples_s_batchN * clip_size

    del model, x1, xb

    return {
        "Params_M": params_m,
        "GMACs_per_sample": gmacs_per_sample,
        "GMACs": gmacs_total,
        "Samples_per_sec_batch1": samples_s_batch1,
        f"Samples_per_sec_batch{batch_size}": samples_s_batchN,
        "FPS_batch1": fps_batch1,
        f"FPS_batch{batch_size}": fps_batchN,
        "Batch_size": batch_size,
        "Clip_size": clip_size
    }


# Image Classifer Speed

In [ ]:
import torch
import warnings
warnings.filterwarnings("ignore") # warnings.resetwarnings()
import gc
gc.collect()
torch.cuda.empty_cache()


model_list = [  (load_resnet, 0, "ResNet-18"), (load_resnet, 1,"ResNet-50"), (load_resnet, 2,"ResNet-152"),
                (load_efficientnet, 0,"EfficientNetV2_S"), (load_efficientnet, 1,"EfficientNetV2_M"),(load_efficientnet, 2,"EfficientNetV2_L"),
                (load_densenet, 0,"DenseNet-121"), (load_densenet, 1,"DenseNet-169"), (load_densenet, 2,"DenseNet-201"),
             ]

for model_func, model_index,model_name in model_list:
    a=evaluate_model_cost( model=model_func(model_index).to(device), input_size=(3, 128, 128), batch_size=128, clip_size=1, n_warmup=20, n_iters=100)
    print(model_name,a)
    gc.collect()
    torch.cuda.empty_cache()

a=evaluate_model_cost( model=load_inceptionnet(0).to(device), input_size=(3, 128, 128), batch_size=128, clip_size=1, n_warmup=20, n_iters=100)
print("InceptionNetV3",a)
gc.collect()
torch.cuda.empty_cache()

# Video Model Speed

In [ ]:
import torch
import torch.nn as nn
import gc
from pytorchvideo.models.hub import x3d_xs,x3d_l, mvit_base_16x4
from torchvision.models.video import r2plus1d_18,r3d_18


class VideoModelX3D(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        backbone = x3d_l(pretrained=True)

        # ❗ Remove the original classification head entirely
        self.features = nn.Sequential(*backbone.blocks[:-1])

        # X3D-XS final feature channels = 192
        self.pool = nn.AdaptiveAvgPool3d((1, 1, 1))
        self.fc = nn.Linear(192, num_classes)

    def forward(self, x):
        """
        x: (B, T, C, H, W)
        """
        x = x.permute(0, 2, 1, 3, 4)   # (B, C, T, H, W)

        x = self.features(x)           # backbone without head
        x = self.pool(x)               # (B, 192, 1, 1, 1)
        x = x.flatten(1)               # (B, 192)
        x = self.fc(x)                 # (B, num_classes)

        return x

class VideoModelMViT(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.backbone = mvit_base_16x4(pretrained=True)

        in_features = self.backbone.head.proj.in_features
        self.backbone.head.proj = nn.Linear(in_features, num_classes)


    def forward(self, x):
        """
        x: (B, T, C, H, W)
        """
        B, T, C, H, W = x.shape
    
        # merge batch and time
        x = x.view(B * T, C, H, W)
    
        # resize frames
        x = F.interpolate(
            x, size=(224, 224),
            mode="bilinear",
            align_corners=False
        )
    
        # restore shape
        x = x.view(B, T, C, 224, 224)
    
        # convert to (B, C, T, H, W)
        x = x.permute(0, 2, 1, 3, 4)
    
        return self.backbone(x)



class VideoModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.backbone = r2plus1d_18(weights="KINETICS400_V1")
        self.backbone.fc = nn.Linear(
            self.backbone.fc.in_features,
            num_classes
        )

    def forward(self, x):
        """
        x: (B, T, C, H, W)
        """
        x = x.permute(0, 2, 1, 3, 4)  # → (B, C, T, H, W)
        return self.backbone(x)
import torch
from torchvision.models.video import mc3_18,r3d_18
import torch.nn as nn

class VideoModelres3(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.backbone = r3d_18(weights="KINETICS400_V1")
        self.backbone.fc = nn.Linear(
            self.backbone.fc.in_features,
            num_classes
        )

    def forward(self, x):
        """
        x: (B, T, C, H, W)
        """
        x = x.permute(0, 2, 1, 3, 4)  # → (B, C, T, H, W)
        return self.backbone(x)

class VideoModelmc3(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.backbone = mc3_18(weights="KINETICS400_V1")
        self.backbone.fc = nn.Linear(
            self.backbone.fc.in_features,
            num_classes
        )

    def forward(self, x):
        """
        x: (B, T, C, H, W)
        """
        x = x.permute(0, 2, 1, 3, 4)  # → (B, C, T, H, W)
        return self.backbone(x)
        


In [ ]:
gc.collect()
torch.cuda.empty_cache()
a=evaluate_model_cost(model=VideoModel(num_classes=7).to(device), input_size=(16,3, 128, 128), clip_size=16,batch_size=8, device="cuda", n_warmup=20, n_iters=100)
print("ResNet(2+1)D18",a)
gc.collect()
torch.cuda.empty_cache()
a=evaluate_model_cost(model=VideoModelMViT(num_classes=7).to(device), input_size=(16,3, 128, 128), clip_size=16,batch_size=8, device="cuda", n_warmup=20, n_iters=100)
print("VideoModelMViT",a)
gc.collect()
torch.cuda.empty_cache()
a=evaluate_model_cost(model=VideoModelX3D(num_classes=7).to(device), input_size=(16,3, 128, 128),clip_size=16,batch_size=8, device="cuda", n_warmup=20, n_iters=100)
print("VideoModelX3D",a)
gc.collect()
torch.cuda.empty_cache()

a=evaluate_model_cost(model=VideoModelres3(num_classes=7).to(device), input_size=(16,3, 128, 128),clip_size=16, batch_size=8, device="cuda", n_warmup=20, n_iters=100)
print("ResNet3D18",a)
gc.collect()
torch.cuda.empty_cache()
a=evaluate_model_cost(model=VideoModelmc3(num_classes=7).to(device), input_size=(16,3, 128, 128), clip_size=16,batch_size=8, device="cuda", n_warmup=20, n_iters=100)
print("MC3-18",a)
gc.collect()
torch.cuda.empty_cache()


# DINOv3 Speed

In [ ]:
# (model_name,weights_path) tuples for "load_dinov3_vit" function arguemtns
dino_list=[("dinov3_vits16","dinov3-main/dinov3_vits16_pretrain_lvd1689m-08c60483.pth"),
          ("dinov3_vits16plus","dinov3-main/dinov3_vits16plus_pretrain_lvd1689m-4057cbaa.pth"),
          ("dinov3_vitb16","dinov3-main/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth"),
          ("dinov3_vitl16","dinov3-main/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth")]


def load_dinov3_vit(model_name, weights_path, num_classes):
    backbone = torch.hub.load(repo_or_dir="dinov3-main", model=model_name, source="local", weights=weights_path)

    feature_dim = backbone.num_features if hasattr(backbone, "num_features") else 1024 
    class DINOv3Classifier(nn.Module):
        def __init__(self, backbone):
            super().__init__()
            self.backbone = backbone
            self.head = nn.Linear(feature_dim, num_classes)

        def forward(self, x):
            feats = self.backbone.forward_features_list([x], [None])[0]
            pooled = feats["x_norm_clstoken"]
            logits = self.head(pooled)
            return logits

    return DINOv3Classifier(backbone)


In [ ]:
for model_name,weights_path in dino_list:
    model= load_dinov3_vit(model_name=model_name,weights_path=weights_path, num_classes=7)
    a=evaluate_model_cost(model=model.to(device), input_size=(3, 128, 128), batch_size=128, device="cuda", n_warmup=20, n_iters=100)
    print(model_name, a)